In [1]:
# 1. जरूरी लाइब्रेरीज इंस्टॉल और इम्पोर्ट करें
!pip install -q transformers datasets accelerate torch
import torch
import gc
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoConfig, 
    AutoModelForCausalLM, 
    Trainer, 
    TrainingArguments, 
    DataCollatorForLanguageModeling
)

print("लॉन्चिंग ZENO AI: स्टेप-बाय-स्टेप (Curriculum Learning) ट्रेनिंग...")

# टोकनाइज़र और Data Collator एक ही बार सेट करेंगे
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# ==========================================
# स्टेज 1: हिंदी-इंग्लिश वोकैबुलरी (Language Foundation)
# ==========================================
print("\n--- स्टेज 1 शुरू: हिंदी-इंग्लिश वोकैबुलरी ---")
ds_hindi = load_dataset("cfilt/iitb-english-hindi", split="train[:1%]")
def format_hindi(example):
    translation = example.get("translation", {})
    return {"text": f"English: {translation.get('en', '')}\nHindi: {translation.get('hi', '')}"}
ds_hindi = ds_hindi.map(format_hindi, remove_columns=ds_hindi.column_names)
tokenized_stage1 = ds_hindi.map(tokenize_function, batched=True)

# बिल्कुल खाली दिमाग (Scratch) से शुरू कर रहे हैं
config = AutoConfig.from_pretrained("gpt2") 
model = AutoModelForCausalLM.from_config(config) 

args_stage1 = TrainingArguments(
    output_dir="/kaggle/working/temp_stage1",
    num_train_epochs=1, per_device_train_batch_size=8,
    save_steps=500, fp16=torch.cuda.is_available(), report_to="none"
)

trainer1 = Trainer(model=model, args=args_stage1, train_dataset=tokenized_stage1, data_collator=data_collator)
trainer1.train()

# स्टेज 1 का मॉडल सेव करें
print("स्टेज 1 पूरा हुआ! मॉडल 'zeno_step1' में सेव हो रहा है...")
model.save_pretrained("/kaggle/working/zeno_step1")
tokenizer.save_pretrained("/kaggle/working/zeno_step1")

# मेमोरी (RAM/GPU) साफ करें ताकि Colab/Kaggle क्रैश न हो
del trainer1, model, ds_hindi, tokenized_stage1
torch.cuda.empty_cache()
gc.collect()

# ==========================================
# स्टेज 2: प्रोग्रामिंग और कोडिंग (Technical Skills)
# ==========================================
print("\n--- स्टेज 2 शुरू: प्रोग्रामिंग ट्रेनिंग ---")
ds_code = load_dataset("flytech/python-codes-25k", split="train[:2%]")
def format_code(example):
    return {"text": f"Task: {example.get('instruction', '')}\nSolution: {example.get('output', '')}"}
ds_code = ds_code.map(format_code, remove_columns=ds_code.column_names)
tokenized_stage2 = ds_code.map(tokenize_function, batched=True)

# ध्यान दें: यहाँ हम 'zeno_step1' से मॉडल लोड कर रहे हैं (ताकि वो पुरानी बातें न भूले)
model = AutoModelForCausalLM.from_pretrained("/kaggle/working/zeno_step1")

args_stage2 = TrainingArguments(
    output_dir="/kaggle/working/temp_stage2",
    num_train_epochs=1, per_device_train_batch_size=8,
    save_steps=500, fp16=torch.cuda.is_available(), report_to="none"
)

trainer2 = Trainer(model=model, args=args_stage2, train_dataset=tokenized_stage2, data_collator=data_collator)
trainer2.train()

print("स्टेज 2 पूरा हुआ! मॉडल 'zeno_step2' में सेव हो रहा है...")
model.save_pretrained("/kaggle/working/zeno_step2")

# मेमोरी फिर से साफ करें
del trainer2, model, ds_code, tokenized_stage2
torch.cuda.empty_cache()
gc.collect()

# ==========================================
# स्टेज 3: राइटिंग और स्टोरीटेलिंग (Creative Skills)
# ==========================================
print("\n--- स्टेज 3 शुरू: राइटिंग और स्टोरीटेलिंग ---")
ds_writing = load_dataset("roneneldan/TinyStories", split="train[:2%]")
tokenized_stage3 = ds_writing.map(tokenize_function, batched=True)

# अब हम 'zeno_step2' को लोड करेंगे, जिसमें भाषा और कोडिंग दोनों का ज्ञान है
model = AutoModelForCausalLM.from_pretrained("/kaggle/working/zeno_step2")

args_stage3 = TrainingArguments(
    output_dir="/kaggle/working/temp_stage3",
    num_train_epochs=1, per_device_train_batch_size=8,
    save_steps=500, fp16=torch.cuda.is_available(), report_to="none"
)

trainer3 = Trainer(model=model, args=args_stage3, train_dataset=tokenized_stage3, data_collator=data_collator)
trainer3.train()

print("स्टेज 3 पूरा हुआ! फाइनल ZENO AI तैयार हो रहा है...")
model.save_pretrained("/kaggle/working/zeno_final_model")
tokenizer.save_pretrained("/kaggle/working/zeno_final_model")

print("🎉 बधाई हो! आपका स्टेप-बाय-स्टेप ट्रेन किया हुआ ZENO AI '/kaggle/working/zeno_final_model' में सेव हो गया है।")


लॉन्चिंग ZENO AI: स्टेप-बाय-स्टेप (Curriculum Learning) ट्रेनिंग...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

स्टेज 3 पूरा हुआ! फाइनल ZENO AI तैयार हो रहा है...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 बधाई हो! आपका स्टेप-बाय-स्टेप ट्रेन किया हुआ ZENO AI '/kaggle/working/zeno_final_model' में सेव हो गया है।


In [2]:
# 1. जरूरी लाइब्रेरीज (अगर पहले से रन नहीं हैं)
!pip install -q transformers datasets accelerate torch
import torch
import gc
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    Trainer, 
    TrainingArguments, 
    DataCollatorForLanguageModeling
)

print("🧠 ZENO AI का कंटीन्यूअस लर्निंग सिस्टम चालू हो रहा है...")

# ==========================================
# ⚙️ सेटिंग्स (यहाँ आप बदलाव कर सकते हैं)
# ==========================================

# पुराना मॉडल कहाँ रखा है? (इसे मत बदलें)
MODEL_PATH = "/kaggle/working/zeno_final_model"

# नया डेटासेट जो आप सिखाना चाहते हैं 
NEW_DATASET_NAME = "cfilt/iitb-english-hindi" 

# ==========================================

# 2. पुराना ZENO AI मॉडल और उसका दिमाग (Tokenizer) लोड करना
print("पुराना ज्ञान लोड किया जा रहा है ताकि कुछ भूले नहीं...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)

# 3. नया डेटासेट इंटरनेट से उठाना और फॉर्मेट करना
print(f"नया डेटा ({NEW_DATASET_NAME}) डाउनलोड हो रहा है...")
dataset = load_dataset(NEW_DATASET_NAME, split="train[:3%]") # 3% हिस्सा ले रहे हैं

# डेटा को सही फॉर्मेट में बदलने का फंक्शन (हिंदी-इंग्लिश के लिए)
def format_data(example):
    if 'translation' in example:
        en = example['translation'].get('en', '')
        hi = example['translation'].get('hi', '')
        return {"text": f"English: {en}\nHindi: {hi}"}
    elif 'text' in example:
        return {"text": example['text']}
    else:
        return {"text": str(example)}

formatted_dataset = dataset.map(format_data, remove_columns=dataset.column_names)

# 4. डेटा को मॉडल की भाषा में बदलना (Tokenization)
print("डेटा को मॉडल के समझने लायक बनाया जा रहा है...")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. ट्रेनिंग की सेटिंग्स (FIXED)
args = TrainingArguments(
    output_dir="/kaggle/working/temp_update",
    num_train_epochs=1, 
    per_device_train_batch_size=4, # Memory (VRAM) बचाने के लिए इसे 8 से 4 कर दिया है
    save_strategy="no",            # ⚠️ FIX: बीच-बीच में सेव करना बंद कर दिया है ताकि Disk Space फुल न हो
    fp16=torch.cuda.is_available(), 
    report_to="none",
    learning_rate=2e-5 
)

trainer = Trainer(
    model=model, 
    args=args, 
    train_dataset=tokenized_dataset, 
    data_collator=data_collator
)

# 6. नई ट्रेनिंग शुरू!
print("🚀 नई ट्रेनिंग शुरू हो रही है...")
trainer.train()

# 7. अपडेटेड मॉडल को वापस उसी नाम से सेव करना
print("✅ ट्रेनिंग पूरी हुई! ZENO AI अब और भी ज्यादा स्मार्ट हो गया है। नया ज्ञान सेव किया जा रहा है...")

# सेव करने से पहले Kaggle का थोड़ा स्पेस खाली करें
import shutil
if os.path.exists("/kaggle/working/temp_update"):
    shutil.rmtree("/kaggle/working/temp_update")

model.save_pretrained(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

# सिस्टम की मेमोरी साफ करना
del trainer, model, dataset, formatted_dataset, tokenized_dataset
torch.cuda.empty_cache()
gc.collect()

print("🎉 बधाई! ZENO AI का नया वर्ज़न तैयार है और सेव हो चुका है।")


🧠 ZENO AI का कंटीन्यूअस लर्निंग सिस्टम चालू हो रहा है...
पुराना ज्ञान लोड किया जा रहा है ताकि कुछ भूले नहीं...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

नया डेटा (cfilt/iitb-english-hindi) डाउनलोड हो रहा है...


Map:   0%|          | 0/49772 [00:00<?, ? examples/s]

डेटा को मॉडल के समझने लायक बनाया जा रहा है...


...

🚀 नई ट्रेनिंग शुरू हो रही है...


Step,Training Loss
500,2.142287
1000,1.923664
1500,1.815927
2000,1.716437
2500,1.661910
3000,1.594474
3500,1.529943
4000,1.504404
4500,1.504506
5000,1.456869


✅ ट्रेनिंग पूरी हुई! ZENO AI अब और भी ज्यादा स्मार्ट हो गया है। नया ज्ञान सेव किया जा रहा है...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 बधाई! ZENO AI का नया वर्ज़न तैयार है और सेव हो चुका है।


In [3]:
# 1. Zaruri Libraries (अब लेटेस्ट वर्ज़न ही इस्तेमाल करेंगे)
!pip install -q transformers datasets accelerate torch
import torch, gc, os, shutil
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    Trainer, 
    TrainingArguments, 
    DataCollatorForLanguageModeling
)

print("🚀 ZENO AI: Engineering & Expert Coding Mode Activation...")

MODEL_PATH = "/kaggle/working/zeno_final_model"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)

# ==========================================
# 📊 EXPERT DATASET LOADING (Parquet Supported)
# ==========================================

# A. Programming Knowledge (Modern Parquet Dataset)
print("1/2: Dunia bhar ki programming languages ka data load ho raha hai...")
# FIX: codeparrot की जगह एक मॉडर्न, Parquet-supported डेटासेट लगा दिया है
ds_pro_code = load_dataset("sahil2801/CodeAlpaca-20k", split="train[:5%]") 

def format_pro_code(example):
    return {"text": f"Task: {example.get('instruction', '')}\nCode:\n{example.get('output', '')}"}

ds_pro_code = ds_pro_code.map(format_pro_code, remove_columns=ds_pro_code.column_names)

# B. Advanced Engineering Mathematics
print("2/2: Engineering Mathematics aur Scientific Logic load ho raha hai...")
# FIX: trust_remote_code=True हटा दिया गया है क्योंकि यह डेटासेट अब सीधे सपोर्टेड है
ds_math = load_dataset("open-web-math/open-web-math", split="train[:1%]")

def format_math(example):
    return {"text": f"Mathematical Theory/Calculation: {example.get('text', '')}"}

ds_math = ds_math.map(format_math, remove_columns=ds_math.column_names)

# C. Mixing Both (Coding + Math)
expert_dataset = concatenate_datasets([ds_pro_code, ds_math]).shuffle(seed=42)

# ==========================================
# 🧠 TRAINING CONFIGURATION
# ==========================================

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_dataset = expert_dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Training Settings
args = TrainingArguments(
    output_dir="/kaggle/working/expert_temp",
    num_train_epochs=2, 
    per_device_train_batch_size=4, 
    gradient_accumulation_steps=4, 
    learning_rate=1e-5, 
    fp16=torch.cuda.is_available(),
    save_strategy="no", # Storage फुल होने से बचाने के लिए 
    report_to="none"
)

trainer = Trainer(
    model=model, 
    args=args, 
    train_dataset=tokenized_dataset, 
    data_collator=data_collator
)

# 🚀 Start Training
print("🛠️ ZENO AI ab ek Engineering Expert ban raha hai... Isme time lagega.")
trainer.train()

# Final save से पहले temp डेटा हटाना
if os.path.exists("/kaggle/working/expert_temp"):
    shutil.rmtree("/kaggle/working/expert_temp")

# Final Save
model.save_pretrained(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

print("🎉 Mubarak Ho! ZENO AI ke paas ab coding aur math ka expert gyan hai.")


🚀 ZENO AI: Engineering & Expert Coding Mode Activation...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

1/2: Dunia bhar ki programming languages ka data load ho raha hai...


README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/64153 [00:00<?, ? examples/s]

🛠️ ZENO AI ab ek Engineering Expert ban raha hai... Isme time lagega.


Step,Training Loss
500,6.772096
1000,6.091748
1500,5.859825
2000,5.722229
2500,5.621755
3000,5.563393
3500,5.524422
4000,5.512981


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Mubarak Ho! ZENO AI ke paas ab coding aur math ka expert gyan hai.
